In [1]:
from pathlib import Path
from met_pipeline.io import load_excel, load_csv, normalize_kab_kota
from met_pipeline.quality import compute_entity_completeness

PROJECT_ROOT = Path.cwd().parent

df = load_excel(
    PROJECT_ROOT / "data" / "raw" / "Data_Timbulan_Sampah_SIPSN_KLHK.xlsx",
    header=0,
)

ref_df = load_csv(
    PROJECT_ROOT / "data" / "reference" / "master_kab_reference_final.csv"
)


In [2]:
df.columns

Index(['tahun', 'provinsi', 'kabupaten/kota', 'timbulan sampah harian(ton)',
       'timbulan sampah tahunan(ton)'],
      dtype='object')

In [3]:
ref_df.columns

Index(['provinsi', 'kab_kota'], dtype='object')

In [5]:
df["kab_kota_key"] = df["kabupaten/kota"].apply(normalize_kab_kota)
ref_df["kab_kota_key"] = ref_df["kab_kota"].apply(normalize_kab_kota)

In [6]:
merged = compute_entity_completeness(
    df=df,
    reference_df=ref_df,
    year_col="tahun",
    province_col="provinsi",
    entity_key_col="kab_kota_key",
    value_col="timbulan sampah tahunan(ton)",
)

In [7]:
merged.head()

,provinsi,kab_kota_key,tahun,timbulan sampah tahunan(ton),completeness
0,Aceh,kabupaten_aceh barat,2024,37432.21,1
1,Aceh,kabupaten_aceh barat,2023,36813.72,1
2,Aceh,kabupaten_aceh barat,2022,37021.59,1
3,Aceh,kabupaten_aceh barat,2021,NaN,0
4,Aceh,kabupaten_aceh barat,2020,NaN,0


In [8]:
merged["completeness"].value_counts()

completeness
0    1887
1    1711
Name: count, dtype: int64

In [9]:
merged.groupby("tahun")["completeness"].mean().sort_index()

tahun
2018    0.001946
2019    0.445525
2020    0.449416
2021    0.443580
2022    0.626459
2023    0.750973
2024    0.610895
Name: completeness, dtype: float64

In [10]:
merged.groupby(["tahun", "provinsi"])["completeness"].mean()

tahun  provinsi         
2018   Aceh                 0.000000
       Bali                 0.000000
       Banten               0.000000
       Bengkulu             0.000000
       DKI Jakarta          0.000000
                              ...   
2024   Sulawesi Tenggara    0.294118
       Sulawesi Utara       0.933333
       Sumatera Barat       0.736842
       Sumatera Selatan     0.529412
       Sumatera Utara       0.303030
Name: completeness, Length: 266, dtype: float64

region muncul di 2022
Kabupaten Sumba Barat Daya	25 Juli 2022
Kabupaten Sumba Tengah	5 Juli 2022